In [ ]:
# November 2, 2026. Run in a fresh kernel in the Studio environment that fails.
REGION = "ca-central-1"
SERVER_NUMBER = 1  # Choose a number from the list below; use 2 to test the other server.


In [ ]:
import os
import time
from importlib.metadata import version
from textwrap import shorten

import boto3
from botocore.config import Config
import mlflow
import sagemaker_mlflow  # The installed plugin signs ordinary MLflow calls for AWS.

session = boto3.Session(region_name=REGION)
client_config = Config(connect_timeout=5, read_timeout=5, retries={"total_max_attempts": 1})
sm = session.client("sagemaker", config=client_config)
caller = session.client("sts", config=client_config).get_caller_identity()
print("CALLER:", caller["Arn"])
print("PLUGIN ROLE OVERRIDE:", os.getenv("SAGEMAKER_MLFLOW_ASSUME_ROLE_ARN") or "none")
print(f"CLIENT: mlflow={version('mlflow')} / plugin={version('sagemaker-mlflow')}")

servers = []
request = {"MaxResults": 100}
while True:
    response = sm.list_mlflow_tracking_servers(**request)
    servers.extend(response["TrackingServerSummaries"])
    if not response.get("NextToken"):
        break
    request["NextToken"] = response["NextToken"]

servers.sort(key=lambda server: server["TrackingServerName"])
for number, server in enumerate(servers, start=1):
    print(f"{number}. {server['TrackingServerName']} | {server['TrackingServerStatus']} | MLflow {server['MlflowVersion']}")
    print("   ", server["TrackingServerArn"])
if not servers:
    print(f"No tracking servers found in {REGION}.")


In [ ]:
if not 1 <= SERVER_NUMBER <= len(servers):
    raise ValueError("Set SERVER_NUMBER to a number printed above.")
tracking_arn = servers[SERVER_NUMBER - 1]["TrackingServerArn"]

# These are HTTP timeouts, not a total wall-clock deadline. No background worker.
os.environ["MLFLOW_HTTP_REQUEST_TIMEOUT"] = "5"
os.environ["MLFLOW_HTTP_REQUEST_MAX_RETRIES"] = "0"
print("SEARCH:", servers[SERVER_NUMBER - 1]["TrackingServerName"], flush=True)
started = time.monotonic()
search_error = None
experiments = None
try:
    mlflow.set_tracking_uri(tracking_arn)
    experiments = mlflow.search_experiments(max_results=1)
    print(f"PASS: returned {len(experiments)} experiment(s) in {time.monotonic() - started:.1f}s")
except Exception as error:
    search_error = error  # Full exception remains available for optional inspection.
    message = " ".join(str(error).split())
    if "ConnectTimeout" in message or "connect timeout" in message.lower():
        detail = "Connection timed out before receiving an HTTP response."
    else:
        detail = shorten(message, width=240, placeholder="...")
    print(f"FAIL: {type(error).__name__} after {time.monotonic() - started:.1f}s")
    print(detail)
